# Test 2: ROC-AUC and Precision-Recall AUC Curves

**Goal**: Compute and plot ROC and Precision-Recall curves for:
- Model A — GraphCodeBERT (with DFG attention)
- Model B — CodeBERT (text-only baseline)
- Ensemble — soft-vote average of A + B

**Why this matters**: Accuracy is misleading in security tasks. A model tuned for
high recall (catching all malware) will have different tradeoffs than one tuned for
precision. ROC-AUC and PR-AUC characterize the *full* operating curve, not just
the threshold-0.5 slice.

**Kaggle inputs**:
- `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`
- `/kaggle/input/model2/model.bin` (GraphCodeBERT weights)
- `/kaggle/input/model-codebert/model_codebert.bin` — upload your CodeBERT checkpoint

**Output**: PNG plots + `/kaggle/working/test2_auc_results.txt`

In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 13.0 MB/s eta 0:00:00


In [2]:
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import RobertaConfig, RobertaModel, AutoTokenizer
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, classification_report
)
import matplotlib
matplotlib.use('Agg')          # headless backend for Kaggle
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Imports OK")
print(f"CUDA: {torch.cuda.is_available()}")

Imports OK
CUDA: True


In [3]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
class Args:
    train_file        = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    gcb_weights       = "/kaggle/input/datasets/hasanmahmudabdullah/model2/model.bin"
    # Update this path to wherever you uploaded the CodeBERT checkpoint:
    codebert_weights  = "/kaggle/input/datasets/hasanmahmudabdullah/codebert/model_codebert.bin"

    code_length       = 384
    data_flow_length  = 128
    eval_batch_size   = 32
    seed              = 42
    val_ratio         = 0.10   # same 90/10 split as original notebook

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu  = torch.cuda.device_count()

args = Args()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if args.n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)
print(f"Device: {args.device}")

Device: cuda


In [4]:
# ─── MODEL A — GraphCodeBERT (DFG-aware) ─────────────────────────────────────
class ModelA(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder    = encoder
        self.config     = config
        self.tokenizer  = tokenizer
        self.args       = args
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        ext_mask = (1.0 - attn_mask) * -10000.0
        ext_mask = ext_mask.unsqueeze(1)
        emb = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)
        out = self.encoder.encoder(
            emb, attention_mask=ext_mask,
            head_mask=[None] * self.config.num_hidden_layers
        )[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob


# ─── MODEL B — CodeBERT (text-only) ──────────────────────────────────────────
class ModelB(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder    = encoder
        self.config     = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob

In [5]:
# ─── DATASET A — with DFG ─────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args      = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        with open(file_path) as f:
            self.lines = f.readlines()

    def __len__(self): return len(self.lines)

    def _char_idx(self, lines, coord):
        r, c = coord
        return sum(len(lines[i]) for i in range(min(r, len(lines)))) + c

    def __getitem__(self, idx):
        entry     = json.loads(self.lines[idx])
        code      = entry.get('code', '')
        dfg       = entry.get('dfg', [])[:self.args.data_flow_length]
        label     = int(entry.get('label', 0))
        code_lines = code.splitlines(keepends=True)

        tok = self.tokenizer(
            code, max_length=self.args.code_length, truncation=True,
            padding='max_length', return_offsets_mapping=True
        )
        input_ids, offsets = tok['input_ids'], tok['offset_mapping']
        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        n2t, p2n = {}, {}

        for ni, node in enumerate(dfg):
            sp, ep = node[1][0], node[1][1]
            pk     = (sp[0], sp[1], ep[0], ep[1])
            p2n[pk] = ni
            try:
                cs, ce = self._char_idx(code_lines, sp), self._char_idx(code_lines, ep)
                n2t[ni] = [i for i, (ts, te) in enumerate(offsets)
                           if ts != te and
                           ((ts >= cs and te <= ce) or (cs >= ts and ce <= te))]
            except IndexError:
                n2t[ni] = []

        c_len = self.args.code_length
        mask  = np.zeros((self.total_len, self.total_len), dtype=bool)
        mask[:c_len, :c_len] = True
        for ni, node in enumerate(dfg):
            ani = c_len + ni
            for ti in n2t.get(ni, []):
                mask[ani, ti] = mask[ti, ani] = True
            for pp in node[4]:
                pk = (pp[0][0], pp[0][1], pp[1][0], pp[1][1])
                if pk in p2n:
                    ap = c_len + p2n[pk]
                    mask[ani, ap] = mask[ap, ani] = True
            mask[ani, ani] = True

        ids   = input_ids + dfg_ids
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        pad   = self.total_len - len(ids)
        if pad > 0:
            ids   += [self.tokenizer.pad_token_id] * pad
            p_ids += [1] * pad

        return {
            'input_ids': torch.tensor(ids,   dtype=torch.long),
            'p_ids':     torch.tensor(p_ids, dtype=torch.long),
            'attn_mask': torch.tensor(mask,  dtype=torch.float),
            'label':     torch.tensor(label, dtype=torch.long)
        }


# ─── DATASET B — text-only ────────────────────────────────────────────────────
class SimpleCodeDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.tokenizer = tokenizer
        self.args      = args
        with open(file_path) as f:
            self.lines = f.readlines()

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        try:
            entry = json.loads(self.lines[idx])
        except Exception:
            return self.__getitem__((idx + 1) % len(self.lines))
        code  = entry.get('code', '')
        label = int(entry.get('label', 0))
        tok   = self.tokenizer(
            code, max_length=self.args.code_length,
            truncation=True, padding='max_length'
        )
        return {
            'input_ids':      torch.tensor(tok['input_ids'],      dtype=torch.long),
            'attention_mask': torch.tensor(tok['attention_mask'], dtype=torch.long),
            'label':          torch.tensor(label,                 dtype=torch.long)
        }

In [6]:
# ─── LOAD MODELS ─────────────────────────────────────────────────────────────
print("Loading Model A (GraphCodeBERT)...")
cfg_a  = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
cfg_a.num_labels = 2
tok_a  = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
enc_a  = RobertaModel.from_pretrained("microsoft/graphcodebert-base", config=cfg_a)
model_a = ModelA(enc_a, cfg_a, tok_a, args).to(args.device)
model_a.load_state_dict(torch.load(args.gcb_weights, map_location=args.device))
model_a.eval()
print("  ✓ Model A loaded")

has_codebert = os.path.exists(args.codebert_weights)
if has_codebert:
    print("Loading Model B (CodeBERT)...")
    cfg_b  = RobertaConfig.from_pretrained("microsoft/codebert-base")
    cfg_b.num_labels = 2
    tok_b  = AutoTokenizer.from_pretrained("microsoft/codebert-base")
    enc_b  = RobertaModel.from_pretrained("microsoft/codebert-base", config=cfg_b)
    model_b = ModelB(enc_b, cfg_b, tok_b, args).to(args.device)
    model_b.load_state_dict(torch.load(args.codebert_weights, map_location=args.device))
    model_b.eval()
    print("  ✓ Model B loaded")
else:
    print(f"⚠  CodeBERT weights not found at {args.codebert_weights}.")
    print("   Only Model A and solo GraphCodeBERT curves will be plotted.")
    model_b = None

Loading Model A (GraphCodeBERT)...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

  ✓ Model B loaded


In [7]:
# ─── PREPARE VALIDATION SETS ─────────────────────────────────────────────────
print("Building datasets...")
ds_a = TextDataset(tok_a, args, args.train_file)
n    = len(ds_a)
train_n = int(0.9 * n)
val_n   = n - train_n

gen = torch.Generator().manual_seed(args.seed)
_, val_ds_a = random_split(ds_a, [train_n, val_n], generator=gen)

if model_b is not None:
    ds_b = SimpleCodeDataset(tok_b, args, args.train_file)
    gen  = torch.Generator().manual_seed(args.seed)
    _, val_ds_b = random_split(ds_b, [train_n, val_n], generator=gen)

loader_a = DataLoader(val_ds_a, batch_size=args.eval_batch_size,
                      shuffle=False, num_workers=2)
if model_b is not None:
    loader_b = DataLoader(val_ds_b, batch_size=args.eval_batch_size,
                          shuffle=False, num_workers=2)

print(f"Val samples: {val_n:,}")

Building datasets...
Val samples: 19,996


In [8]:
# ─── COLLECT RAW PROBABILITIES ─────────────────────────────────────────────
@torch.no_grad()
def get_probs_a(model, loader):
    """Returns (probs_class1, true_labels) numpy arrays."""
    all_probs, all_labels = [], []
    for batch in tqdm(loader, desc="Inference A"):
        inp = {
            'input_ids': batch['input_ids'].to(args.device),
            'p_ids':     batch['p_ids'].to(args.device),
            'attn_mask': batch['attn_mask'].to(args.device)
        }
        probs = model(**inp)          # [B, 2]
        all_probs.extend(probs[:, 1].cpu().numpy())   # probability of class 1
        all_labels.extend(batch['label'].numpy())
    return np.array(all_probs), np.array(all_labels)


@torch.no_grad()
def get_probs_b(model, loader):
    all_probs, all_labels = [], []
    for batch in tqdm(loader, desc="Inference B"):
        inp = {
            'input_ids':      batch['input_ids'].to(args.device),
            'attention_mask': batch['attention_mask'].to(args.device)
        }
        probs = model(**inp)
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_labels.extend(batch['label'].numpy())
    return np.array(all_probs), np.array(all_labels)


probs_a, labels_a = get_probs_a(model_a, loader_a)

if model_b is not None:
    probs_b, labels_b = get_probs_b(model_b, loader_b)
    # Ensemble: average P(class=1) from both models
    probs_ens = (probs_a + probs_b) / 2.0
    print("Collected probs for A, B, and Ensemble.")
else:
    print("Collected probs for Model A only.")

Inference B: 100%|██████████| 625/625 [03:52<00:00,  2.69it/s]

Collected probs for A, B, and Ensemble.


In [9]:
# ─── COMPUTE AUC METRICS ──────────────────────────────────────────────────────
def compute_curves(probs, labels, name):
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc     = auc(fpr, tpr)

    precision, recall, _ = precision_recall_curve(labels, probs)
    pr_auc               = average_precision_score(labels, probs)

    preds_05 = (probs >= 0.5).astype(int)
    acc_05   = accuracy_score(labels, preds_05)

    print(f"\n{name}")
    print(f"  ROC-AUC : {roc_auc:.4f}")
    print(f"  PR-AUC  : {pr_auc:.4f}")
    print(f"  Acc@0.5 : {acc_05:.4%}")
    return fpr, tpr, roc_auc, precision, recall, pr_auc


results = {}
results['GraphCodeBERT'] = compute_curves(probs_a, labels_a, "GraphCodeBERT")
if model_b is not None:
    results['CodeBERT']  = compute_curves(probs_b, labels_b, "CodeBERT")
    results['Ensemble']  = compute_curves(probs_ens, labels_a, "Ensemble (A+B)")


GraphCodeBERT
  ROC-AUC : 0.9798
  PR-AUC  : 0.9797
  Acc@0.5 : 91.8184%

CodeBERT
  ROC-AUC : 0.9745
  PR-AUC  : 0.9745
  Acc@0.5 : 90.4431%

Ensemble (A+B)
  ROC-AUC : 0.9804
  PR-AUC  : 0.9803
  Acc@0.5 : 91.8684%


In [10]:
# ─── PLOT ROC CURVES ─────────────────────────────────────────────────────────
colors = {'GraphCodeBERT': '#4C72B0', 'CodeBERT': '#DD8452', 'Ensemble': '#55A868'}

fig, ax = plt.subplots(figsize=(8, 6))

for name, (fpr, tpr, roc_auc, *_) in results.items():
    ax.plot(fpr, tpr, color=colors[name], lw=2,
            label=f"{name}  (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5000)')
ax.set_xlim([-0.01, 1.0])
ax.set_ylim([0.0, 1.01])
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate',  fontsize=13)
ax.set_title('ROC Curves — GraphCodeBERT vs CodeBERT vs Ensemble', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

roc_path = "/kaggle/working/test2_roc_curve.png"
fig.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"ROC curve saved → {roc_path}")

ROC curve saved → /kaggle/working/test2_roc_curve.png


In [11]:
# ─── PLOT PRECISION-RECALL CURVES ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for name, (_, _, roc_auc, precision, recall, pr_auc) in results.items():
    ax.plot(recall, precision, color=colors[name], lw=2,
            label=f"{name}  (AP = {pr_auc:.4f})")

# Baseline: random classifier = class prevalence
prevalence = float(labels_a.mean())
ax.axhline(y=prevalence, color='k', linestyle='--', lw=1,
           label=f'Random baseline (AP = {prevalence:.4f})')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.01])
ax.set_xlabel('Recall',    fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curves — GraphCodeBERT vs CodeBERT vs Ensemble', fontsize=13)
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, alpha=0.3)

pr_path = "/kaggle/working/test2_pr_curve.png"
fig.savefig(pr_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"PR curve saved → {pr_path}")

PR curve saved → /kaggle/working/test2_pr_curve.png


In [12]:
# ─── THRESHOLD SENSITIVITY (GraphCodeBERT) ───────────────────────────────────
# In APK security, missing malware (FN) is worse than a false alarm (FP).
# Show how precision and recall change as we lower the decision threshold.

thresholds = np.arange(0.1, 0.9, 0.05)

print("\nTarget: GraphCodeBERT — Threshold Sensitivity")
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Accuracy':>10}")
print("-" * 55)

th_records = []
for th in thresholds:
    preds  = (probs_a >= th).astype(int)
    from sklearn.metrics import precision_score, recall_score, f1_score
    prec   = precision_score(labels_a, preds, zero_division=0)
    rec    = recall_score(labels_a, preds, zero_division=0)
    f1     = f1_score(labels_a, preds, zero_division=0)
    acc    = accuracy_score(labels_a, preds)
    print(f"  {th:.2f}     {prec:.4f}    {rec:.4f}    {f1:.4f}    {acc:.4%}")
    th_records.append((th, prec, rec, f1, acc))

# Best F1 threshold
best = max(th_records, key=lambda x: x[3])
print(f"\nBest F1 at threshold={best[0]:.2f}: "
      f"F1={best[3]:.4f}  Prec={best[1]:.4f}  Rec={best[2]:.4f}")


Target: GraphCodeBERT — Threshold Sensitivity
 Threshold  Precision     Recall         F1   Accuracy
-------------------------------------------------------
  0.10     0.7988    0.9915    0.8848    87.2174%
  0.15     0.8207    0.9859    0.8958    88.6377%
  0.20     0.8372    0.9813    0.9035    89.6229%
  0.25     0.8508    0.9765    0.9093    90.3531%
  0.30     0.8622    0.9705    0.9131    90.8582%
  0.35     0.8732    0.9626    0.9157    91.2282%
  0.40     0.8863    0.9503    0.9172    91.5033%
  0.45     0.9020    0.9350    0.9182    91.7483%
  0.50     0.9183    0.9163    0.9173    91.8184%
  0.55     0.9346    0.8944    0.9140    91.6683%
  0.60     0.9501    0.8707    0.9087    91.3333%
  0.65     0.9598    0.8499    0.9015    90.8082%
  0.70     0.9662    0.8346    0.8956    90.3631%
  0.75     0.9710    0.8205    0.8894    89.8980%
  0.80     0.9741    0.8059    0.8820    89.3279%
  0.85     0.9773    0.7909    0.8743    88.7377%

Best F1 at threshold=0.45: F1=0.9182  Pre

In [13]:
# ─── SAVE SUMMARY ────────────────────────────────────────────────────────────
out_path = "/kaggle/working/test2_auc_results.txt"
with open(out_path, "w") as f:
    f.write("Test 2: ROC-AUC and PR-AUC Results\n")
    f.write("=" * 50 + "\n")
    for name, (fpr, tpr, roc_auc, prec, rec, pr_auc) in results.items():
        preds_05 = (probs_a if name == 'GraphCodeBERT'
                    else probs_b if name == 'CodeBERT'
                    else probs_ens) >= 0.5
        acc_05   = accuracy_score(labels_a, preds_05.astype(int))
        f.write(f"\n{name}:\n")
        f.write(f"  ROC-AUC : {roc_auc:.4f}\n")
        f.write(f"  PR-AUC  : {pr_auc:.4f}\n")
        f.write(f"  Acc@0.5 : {acc_05:.4%}\n")
    f.write(f"\nBest F1 Threshold for GraphCodeBERT: {best[0]:.2f} "
            f"(F1={best[3]:.4f})\n")

print(f"Results saved → {out_path}")
print(f"ROC plot   → /kaggle/working/test2_roc_curve.png")
print(f"PR  plot   → /kaggle/working/test2_pr_curve.png")

Results saved → /kaggle/working/test2_auc_results.txt
ROC plot   → /kaggle/working/test2_roc_curve.png
PR  plot   → /kaggle/working/test2_pr_curve.png


In [14]:
# ─── PLOT CONFIDENCE CALIBRATION HISTOGRAM ──────────────────────────────
import seaborn as sns

# We plot the confidence distribution for Correct vs Incorrect predictions
preds_a = (probs_a >= 0.45).astype(int)
correct = (preds_a == labels_a)
incorrect = (preds_a != labels_a)

fig, ax = plt.subplots(figsize=(8, 6))

# Seaborn KDE plots for smooth density
sns.histplot(probs_a[correct], color='#55A868', label='Correct Predictions (TP & TN)', 
             kde=True, stat='density', alpha=0.4, linewidth=0, bins=50, ax=ax)
sns.histplot(probs_a[incorrect], color='#C44E52', label='Incorrect Predictions (FP & FN)', 
             kde=True, stat='density', alpha=0.4, linewidth=0, bins=50, ax=ax)

ax.set_xlim([-0.01, 1.01])
ax.set_xlabel('Model Confidence Score P(Malicious)', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title('GraphCodeBERT Confidence Calibration at Threshold 0.45', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

hist_path = '/kaggle/working/test2_confidence_histogram.png'
fig.savefig(hist_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Confidence Calibration Histogram saved -> {hist_path}")

Confidence Calibration Histogram saved -> /kaggle/working/test2_confidence_histogram.png


In [15]:
from sklearn.metrics import confusion_matrix
preds_b = (probs_b >= 0.50).astype(int)
fn_b = confusion_matrix(labels_b, preds_b)[1, 0]
print(f"CodeBERT (Text-Only) False Negatives: {fn_b:,}")


CodeBERT (Text-Only) False Negatives: 659
